# 🎯 MiniPlaces Scene Recognition - Comprehensive Experiments

## BMI/CS 771 Assignment 2

This notebook systematically improves upon the baseline SimpleNet using techniques from the Neural Network Training lecture.

### 📊 What We'll Achieve:
| Model | Expected Accuracy | Improvement |
|-------|------------------|-------------|
| Baseline SimpleNet | ~35-40% | - |
| + BatchNorm | ~42-47% | +5-7% |
| + Skip Connections | ~48-53% | +6-8% |
| + All Improvements | ~50-55% | +15-20% |

### 🔬 Techniques Covered:
1. **Batch Normalization** - Faster training, implicit regularization
2. **Skip Connections (ResNet)** - Better gradient flow
3. **Stochastic Depth (DropPath)** - Regularization for deep networks
4. **Learning Rate Schedules** - Cosine annealing vs step decay
5. **Data Augmentation** - Color jitter, rotation, random crops

---

# 🔧 Part 1: Environment Setup

First, let's set up our Colab environment and check GPU availability.

In [ ]:
#@title 1.1 Check GPU and System Info
#@markdown Run this cell to verify GPU is available

import torch
import sys
import os

print("=" * 60)
print("SYSTEM INFORMATION")
print("=" * 60)
print(f"Python version: {sys.version.split()[0]}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    
    # Check if A100
    gpu_name = torch.cuda.get_device_name(0)
    if 'A100' in gpu_name:
        print("\n✅ A100 GPU detected! Estimated training time: 8-15 min per experiment")
    elif 'T4' in gpu_name:
        print("\n⚠️ T4 GPU detected. Estimated training time: 30-60 min per experiment")
    elif 'V100' in gpu_name:
        print("\n✅ V100 GPU detected! Estimated training time: 15-25 min per experiment")
else:
    print("\n❌ No GPU detected! Training will be very slow.")
    print("Go to Runtime → Change runtime type → Select GPU")

In [ ]:
#@title 1.2 Mount Google Drive (for saving results)
#@markdown This allows us to save trained models and results

from google.colab import drive
drive.mount('/content/drive')

# Create a directory for this assignment
WORK_DIR = '/content/drive/MyDrive/CS771_Assignment2'
os.makedirs(WORK_DIR, exist_ok=True)
print(f"\n✅ Working directory: {WORK_DIR}")

In [ ]:
#@title 1.3 Upload Assignment Files
#@markdown Upload your assignment2_release.zip file

import zipfile
from google.colab import files

# Check if already extracted
if os.path.exists('/content/assignment2_release'):
    print("✅ Assignment files already exist!")
else:
    print("Please upload assignment2_release.zip")
    uploaded = files.upload()
    
    # Extract the zip file
    for filename in uploaded.keys():
        if filename.endswith('.zip'):
            with zipfile.ZipFile(filename, 'r') as zip_ref:
                zip_ref.extractall('/content/')
            print(f"\n✅ Extracted {filename}")

# Change to code directory
os.chdir('/content/assignment2_release/code')
print(f"\n📁 Current directory: {os.getcwd()}")
print(f"\n📄 Files:")
!ls -la

In [ ]:
#@title 1.4 Install Dependencies
#@markdown Install any missing packages

!pip install tensorboard pyyaml opencv-python-headless -q
print("✅ Dependencies installed!")

---

# 📚 Part 2: Understanding the Baseline

Before improving, let's understand what we're starting with.

In [ ]:
#@title 2.1 View Baseline SimpleNet Architecture
#@markdown Let's examine the original SimpleNet structure

import torch.nn as nn

# This is the ORIGINAL SimpleNet (what we're improving)
class SimpleNet(nn.Module):
    """
    BASELINE MODEL - Original SimpleNet
    
    Architecture:
    - Conv1: 7x7 conv (3→64), stride 2, + MaxPool → 32x32
    - Conv2: Bottleneck (64→64→64→256) + MaxPool → 16x16
    - Conv3: Bottleneck (256→128→128→512) → 16x16
    - Global Average Pool → FC(512→100)
    
    PROBLEMS:
    ❌ No Batch Normalization → unstable training, slow convergence
    ❌ No Skip Connections → poor gradient flow in deeper networks
    ❌ No Regularization → potential overfitting
    """
    def __init__(self, num_classes=100):
        super(SimpleNet, self).__init__()
        self.features = nn.Sequential(
            # conv1: 7x7 conv
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
            # conv2: bottleneck
            nn.Conv2d(64, 64, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 256, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
            # conv3: bottleneck
            nn.Conv2d(256, 128, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 512, kernel_size=1),
            nn.ReLU(inplace=True),
        )
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

# Count parameters
model = SimpleNet()
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("BASELINE SIMPLENET")
print("=" * 50)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"\nModel structure:")
print(model)

## 🎓 Key Improvements from Lecture

### 1. Batch Normalization
```
Conv → BatchNorm → ReLU (instead of Conv → ReLU)
```
- Normalizes activations to zero mean, unit variance
- **Benefits**: Faster convergence, allows higher learning rates, implicit regularization

### 2. Skip Connections (ResNet-style)
```
output = F(x) + x  (identity shortcut)
```
- Creates "gradient highways" for better backpropagation
- **Benefits**: Can train much deeper networks, easier optimization

### 3. Stochastic Depth (DropPath)
```
output = x + drop_path(F(x))  # randomly drop entire blocks during training
```
- Regularization technique for deep networks
- **Benefits**: Prevents overfitting, trains implicit ensemble

### 4. He Initialization
```python
nn.init.kaiming_normal_(weight, mode='fan_out', nonlinearity='relu')
```
- Proper weight scaling for ReLU networks
- **Benefits**: Stable training from the start

---

# 🏗️ Part 3: Improved Architectures

Now let's implement the improved models step by step.

In [ ]:
#@title 3.1 DropPath Implementation (Stochastic Depth)
#@markdown This is used for regularization in deep networks

def drop_path(x, drop_prob: float = 0., training: bool = False):
    """
    Drop paths (Stochastic Depth) per sample.
    
    During training:
    - Randomly drop entire residual branches with probability drop_prob
    - Scale remaining activations by 1/(1-drop_prob)
    
    During inference:
    - Use full network (no dropping)
    """
    if drop_prob == 0. or not training:
        return x
    keep_prob = 1 - drop_prob
    # Create random tensor: 1 = keep, 0 = drop
    shape = (x.shape[0],) + (1,) * (x.ndim - 1)  # (batch_size, 1, 1, 1)
    random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
    random_tensor.floor_()  # binarize
    # Scale by keep_prob to maintain expected value
    output = x.div(keep_prob) * random_tensor
    return output


class DropPath(nn.Module):
    """Drop paths (Stochastic Depth) module."""
    def __init__(self, drop_prob=0.):
        super(DropPath, self).__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        return drop_path(x, self.drop_prob, self.training)

print("✅ DropPath implemented!")
print("\nUsage: Randomly drops entire residual branches during training")
print("Example: drop_prob=0.1 means 10% chance of dropping each block")

In [ ]:
#@title 3.2 SimpleNetBN - Adding Batch Normalization
#@markdown First improvement: Add BatchNorm after each Conv layer

class SimpleNetBN(nn.Module):
    """
    IMPROVEMENT #1: SimpleNet + Batch Normalization
    
    Changes from baseline:
    ✅ Added BatchNorm2d after each Conv2d (before ReLU)
    ✅ Removed bias from Conv2d (BatchNorm has its own bias)
    
    Why this helps:
    - Normalizes activations → more stable gradients
    - Allows higher learning rates → faster training
    - Acts as regularization → better generalization
    
    Expected improvement: +2-5% accuracy
    """
    def __init__(self, num_classes=100):
        super(SimpleNetBN, self).__init__()
        
        self.features = nn.Sequential(
            # Conv1: 7x7 with BatchNorm
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),  # bias=False!
            nn.BatchNorm2d(64),  # ← NEW: BatchNorm BEFORE ReLU
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
            
            # Conv2 Bottleneck with BatchNorm
            nn.Conv2d(64, 64, kernel_size=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 256, kernel_size=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
            
            # Conv3 Bottleneck with BatchNorm
            nn.Conv2d(256, 128, kernel_size=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 512, kernel_size=1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
        )
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)
        
        # Proper initialization
        self._initialize_weights()
    
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                # He initialization for ReLU
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

# Compare parameters
model_bn = SimpleNetBN()
params_bn = sum(p.numel() for p in model_bn.parameters())

print("SIMPLENET WITH BATCHNORM")
print("=" * 50)
print(f"Parameters: {params_bn:,} (vs baseline {total_params:,})")
print(f"\n📈 Expected improvement: +2-5% accuracy")

In [ ]:
#@title 3.3 ResidualBlock - Skip Connections
#@markdown Building block for ResNet-style architecture

class ResidualBlock(nn.Module):
    """
    IMPROVEMENT #2: Residual Block with Skip Connection
    
    Formula: output = F(x) + shortcut(x)
    
    Where:
    - F(x) = Conv → BN → ReLU → Conv → BN
    - shortcut(x) = x (if same dims) or 1x1 conv (if different dims)
    
    Why this helps:
    - Gradients flow directly through skip connection
    - Easier to learn identity mapping
    - Enables training of much deeper networks
    """
    def __init__(self, in_channels, out_channels, stride=1, drop_path_rate=0.0):
        super(ResidualBlock, self).__init__()
        
        # Main path: Conv → BN → ReLU → Conv → BN
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # Stochastic Depth (DropPath) for regularization
        self.drop_path = DropPath(drop_path_rate) if drop_path_rate > 0 else nn.Identity()
        
        # Shortcut: identity if dims match, 1x1 projection otherwise
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1,
                          stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        # Save input for skip connection
        identity = self.shortcut(x)
        
        # Main path
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        
        # Apply stochastic depth (drops entire block with some probability)
        out = self.drop_path(out)
        
        # Skip connection: ADD input to output
        out = out + identity  # ← THE KEY INNOVATION!
        out = self.relu(out)
        
        return out

print("✅ ResidualBlock implemented!")
print("\nKey feature: output = F(x) + x")
print("This allows gradients to flow directly backward through the skip connection.")

In [ ]:
#@title 3.4 ImprovedCNN - Full ResNet-style Architecture
#@markdown Combines all improvements: BatchNorm + Skip Connections + DropPath

class ImprovedCNN(nn.Module):
    """
    BEST MODEL: Combines ALL improvements from lecture
    
    Improvements over baseline:
    ✅ Batch Normalization - faster training, regularization
    ✅ Skip Connections (ResNet) - better gradient flow
    ✅ Stochastic Depth (DropPath) - prevents overfitting
    ✅ He Initialization - proper weight scaling
    ✅ Dropout before FC - additional regularization
    
    Architecture:
    - Stem: 7x7 conv + pool → 64 channels, 32x32
    - Stage 1: 2 residual blocks, 64 channels
    - Stage 2: 2 residual blocks, 128 channels (stride 2)
    - Stage 3: 2 residual blocks, 256 channels (stride 2)
    - Stage 4: 2 residual blocks, 512 channels (stride 2)
    - GAP + Dropout + FC
    
    Total: ~18 conv layers
    Expected improvement: +10-15% over baseline
    """
    def __init__(self, num_classes=100, drop_path_rate=0.1):
        super(ImprovedCNN, self).__init__()
        
        # Stem: initial feature extraction
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )  # 128x128 → 32x32
        
        # Stochastic depth: linearly increase drop rate with depth
        # Early layers: low drop rate (important features)
        # Deep layers: high drop rate (more regularization)
        total_blocks = 8
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, total_blocks)]
        
        # Residual stages
        self.layer1 = self._make_layer(64, 64, 2, stride=1, drop_rates=dpr[0:2])
        self.layer2 = self._make_layer(64, 128, 2, stride=2, drop_rates=dpr[2:4])   # 32→16
        self.layer3 = self._make_layer(128, 256, 2, stride=2, drop_rates=dpr[4:6])  # 16→8
        self.layer4 = self._make_layer(256, 512, 2, stride=2, drop_rates=dpr[6:8])  # 8→4
        
        # Classifier
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(0.2)  # Additional regularization
        self.fc = nn.Linear(512, num_classes)
        
        # Initialize weights properly
        self._initialize_weights()

    def _make_layer(self, in_ch, out_ch, num_blocks, stride, drop_rates):
        """Create a stage of residual blocks."""
        layers = []
        # First block may downsample
        layers.append(ResidualBlock(in_ch, out_ch, stride, drop_rates[0]))
        # Remaining blocks
        for i in range(1, num_blocks):
            layers.append(ResidualBlock(out_ch, out_ch, 1, drop_rates[i]))
        return nn.Sequential(*layers)

    def _initialize_weights(self):
        """He initialization for ReLU networks."""
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        # Stem
        x = self.stem(x)
        
        # Residual stages
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        
        # Classifier
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        x = self.fc(x)
        
        return x

# Compare parameters
model_improved = ImprovedCNN()
params_improved = sum(p.numel() for p in model_improved.parameters())

print("IMPROVED CNN (ResNet-style)")
print("=" * 50)
print(f"Parameters: {params_improved:,}")
print(f"\nComparison:")
print(f"  Baseline SimpleNet: {total_params:,} params")
print(f"  SimpleNet + BN:     {params_bn:,} params")
print(f"  ImprovedCNN:        {params_improved:,} params")
print(f"\n📈 Expected improvement: +10-15% accuracy over baseline!")

In [ ]:
#@title 3.5 Test Model Forward Pass
#@markdown Verify all models work correctly

# Create dummy input (batch_size=2, channels=3, height=128, width=128)
dummy_input = torch.randn(2, 3, 128, 128)

models_to_test = [
    ("SimpleNet (Baseline)", SimpleNet()),
    ("SimpleNetBN (+BatchNorm)", SimpleNetBN()),
    ("ImprovedCNN (Full ResNet-style)", ImprovedCNN()),
]

print("Testing forward pass for all models...")
print("=" * 60)

for name, model in models_to_test:
    model.eval()
    with torch.no_grad():
        output = model(dummy_input)
    print(f"✅ {name}")
    print(f"   Input:  {list(dummy_input.shape)}")
    print(f"   Output: {list(output.shape)} (batch_size, num_classes)")
    print()

---

# 🚀 Part 4: Update modeling.py and Run Training

Now let's update the actual code files and run experiments.

In [ ]:
#@title 4.1 Update modeling.py with Improved Architectures
#@markdown This adds our improved models to the codebase

IMPROVED_CODE = '''
################################################################################
# Part I.2: Design your own CNN - IMPROVED ARCHITECTURES
################################################################################


class SimpleNetBN(nn.Module):
    """SimpleNet with Batch Normalization. Expected: +2-5% accuracy."""
    def __init__(self, conv_op=nn.Conv2d, num_classes=100):
        super(SimpleNetBN, self).__init__()
        self.features = nn.Sequential(
            conv_op(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
            conv_op(64, 64, kernel_size=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            conv_op(64, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            conv_op(64, 256, kernel_size=1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
            conv_op(256, 128, kernel_size=1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            conv_op(128, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            conv_op(128, 512, kernel_size=1, bias=False),
            nn.BatchNorm2d(512), nn.ReLU(inplace=True),
        )
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1.0)
                nn.init.constant_(m.bias, 0.0)

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)


class ResidualBlock(nn.Module):
    """Residual block with skip connection and optional DropPath."""
    def __init__(self, in_ch, out_ch, stride=1, drop_path_rate=0.0):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.drop_path = DropPath(drop_path_rate) if drop_path_rate > 0 else nn.Identity()
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride, bias=False),
                nn.BatchNorm2d(out_ch))

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.drop_path(out)
        return self.relu(out + identity)


class ImprovedCNN(nn.Module):
    """ResNet-style CNN with BatchNorm + Skip Connections + DropPath. Expected: +10-15% accuracy."""
    def __init__(self, num_classes=100, drop_path_rate=0.1):
        super(ImprovedCNN, self).__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 7, 2, 3, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(3, 2, 1))
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, 8)]
        self.layer1 = self._make_layer(64, 64, 2, 1, dpr[0:2])
        self.layer2 = self._make_layer(64, 128, 2, 2, dpr[2:4])
        self.layer3 = self._make_layer(128, 256, 2, 2, dpr[4:6])
        self.layer4 = self._make_layer(256, 512, 2, 2, dpr[6:8])
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(512, num_classes)
        self._init_weights()

    def _make_layer(self, in_ch, out_ch, n, stride, dpr):
        layers = [ResidualBlock(in_ch, out_ch, stride, dpr[0])]
        for i in range(1, n):
            layers.append(ResidualBlock(out_ch, out_ch, 1, dpr[i]))
        return nn.Sequential(*layers)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        return self.fc(x)


# SELECT YOUR MODEL HERE:
# Options: SimpleNet, SimpleNetBN, ImprovedCNN
default_cnn_model = ImprovedCNN
'''

# Read current modeling.py
with open('cls_libs/modeling.py', 'r') as f:
    content = f.read()

# Find and replace the Part I.2 section
import re
pattern = r'(################################################################################\n# Part I\.2: Design your own CNN[\s\S]*?)(################################################################################\n# Part I\.3)'
replacement = IMPROVED_CODE + '\n\n\\2'

new_content = re.sub(pattern, replacement, content)

# Write back
with open('cls_libs/modeling.py', 'w') as f:
    f.write(new_content)

print("✅ modeling.py updated with improved architectures!")
print("\nAvailable models:")
print("  - SimpleNet (baseline)")
print("  - SimpleNetBN (+BatchNorm)")
print("  - ImprovedCNN (ResNet-style, currently selected)")

In [ ]:
#@title 4.2 Create Experiment Config Files
#@markdown Create YAML configs for different experiments

import yaml

# Base config (what all experiments share)
BASE_CONFIG = {
    "output_folder": "../logs",
    "model_type": "CNN",
    "dataset": {"root_folder": "../data"},
    "loader": {"batch_size": 512, "num_workers": 4},
    "input": {
        "img_min_size": 144,
        "img_crop_size": 128,
        "img_color_jitter": 0.15,
        "img_rotation": 15,
        "img_mean": [0.485, 0.456, 0.406],
        "img_std": [0.229, 0.224, 0.225]
    },
    "opt": {
        "type": "SGD",
        "momentum": 0.9,
        "weight_decay": 1e-4,
        "learning_rate": 0.1,
        "epochs": 60,
        "warmup": True,
        "warmup_epochs": 1,
        "schedule_type": "cosine",
        "schedule_steps": [],
        "schedule_gamma": 0.1
    }
}

# Experiment configs
experiments = {
    "exp_baseline": {
        "description": "Baseline (default settings)",
        "changes": {}
    },
    "exp_strong_aug": {
        "description": "Stronger data augmentation",
        "changes": {
            "input": {
                "img_min_size": 160,
                "img_color_jitter": 0.3,
                "img_rotation": 25
            }
        }
    },
    "exp_higher_lr": {
        "description": "Higher LR with longer warmup",
        "changes": {
            "opt": {
                "learning_rate": 0.2,
                "warmup_epochs": 5
            }
        }
    },
    "exp_more_wd": {
        "description": "Stronger weight decay",
        "changes": {
            "opt": {"weight_decay": 5e-4}
        }
    },
    "exp_best": {
        "description": "Best combined settings",
        "changes": {
            "input": {
                "img_min_size": 160,
                "img_color_jitter": 0.25,
                "img_rotation": 20
            },
            "opt": {
                "learning_rate": 0.15,
                "weight_decay": 3e-4,
                "warmup_epochs": 3,
                "epochs": 80
            }
        }
    }
}

# Create config files
import copy

def deep_update(base, updates):
    """Recursively update nested dict."""
    result = copy.deepcopy(base)
    for key, value in updates.items():
        if key in result and isinstance(result[key], dict) and isinstance(value, dict):
            result[key] = deep_update(result[key], value)
        else:
            result[key] = value
    return result

print("Creating experiment config files...")
print("=" * 60)

for exp_name, exp_info in experiments.items():
    config = deep_update(BASE_CONFIG, exp_info["changes"])
    filepath = f"configs/{exp_name}.yaml"
    
    with open(filepath, 'w') as f:
        f.write(f"# {exp_info['description']}\n")
        yaml.dump(config, f, default_flow_style=False)
    
    print(f"✅ Created: {filepath}")
    print(f"   Description: {exp_info['description']}")

print("\n" + "=" * 60)
print("All config files created!")

---

# 🏃 Part 5: Run Experiments

Now let's run the actual training experiments and track results.

In [ ]:
#@title 5.1 Results Tracker
#@markdown Helper class to track and compare experiment results

import json
from datetime import datetime

class ExperimentTracker:
    """Track and compare experiment results."""
    
    def __init__(self, save_path="experiment_results.json"):
        self.save_path = save_path
        self.results = self._load_results()
    
    def _load_results(self):
        if os.path.exists(self.save_path):
            with open(self.save_path, 'r') as f:
                return json.load(f)
        return {}
    
    def save_results(self):
        with open(self.save_path, 'w') as f:
            json.dump(self.results, f, indent=2)
    
    def add_result(self, name, train_acc, val_acc, train_loss, val_loss, epochs, notes=""):
        self.results[name] = {
            "train_acc": train_acc,
            "val_acc": val_acc,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "epochs": epochs,
            "notes": notes,
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M")
        }
        self.save_results()
    
    def show_comparison(self):
        if not self.results:
            print("No results yet. Run some experiments first!")
            return
        
        print("\n" + "=" * 80)
        print("EXPERIMENT RESULTS COMPARISON")
        print("=" * 80)
        print(f"{'Experiment':<25} {'Val Acc':>10} {'Train Acc':>10} {'Δ vs Baseline':>15}")
        print("-" * 80)
        
        baseline_acc = self.results.get("baseline", {}).get("val_acc", 0)
        
        # Sort by val_acc
        sorted_results = sorted(self.results.items(), 
                               key=lambda x: x[1].get('val_acc', 0), 
                               reverse=True)
        
        for name, data in sorted_results:
            val_acc = data.get('val_acc', 0)
            train_acc = data.get('train_acc', 0)
            delta = val_acc - baseline_acc if baseline_acc > 0 else 0
            delta_str = f"+{delta:.1f}%" if delta > 0 else f"{delta:.1f}%"
            
            print(f"{name:<25} {val_acc:>9.1f}% {train_acc:>9.1f}% {delta_str:>15}")
        
        print("=" * 80)

# Initialize tracker
tracker = ExperimentTracker()
print("✅ Experiment tracker initialized!")
print("\nUsage:")
print("  tracker.add_result('exp_name', train_acc, val_acc, train_loss, val_loss, epochs)")
print("  tracker.show_comparison()")

In [ ]:
#@title 5.2 Run Baseline Experiment (SimpleNet with Improvements)
#@markdown This runs training with the ImprovedCNN architecture

#@markdown ---
#@markdown **Training Settings:**
config_file = "configs/exp_baseline.yaml" #@param ["configs/exp_baseline.yaml", "configs/exp_strong_aug.yaml", "configs/exp_higher_lr.yaml", "configs/exp_more_wd.yaml", "configs/exp_best.yaml"]
experiment_name = "baseline" #@param {type:"string"}
print_freq = 50 #@param {type:"integer"}
checkpoint_freq = 10 #@param {type:"integer"}

print(f"\n{'='*60}")
print(f"RUNNING EXPERIMENT: {experiment_name}")
print(f"Config: {config_file}")
print(f"{'='*60}\n")

# Run training
!python train_cls.py {config_file} --output {experiment_name} --print-freq {print_freq} --ckpt-freq {checkpoint_freq}

print(f"\n{'='*60}")
print(f"EXPERIMENT COMPLETE: {experiment_name}")
print(f"{'='*60}")

In [ ]:
#@title 5.3 Record Experiment Results
#@markdown Enter the results from the training output above

#@markdown ---
#@markdown **Enter Final Results:**
exp_name = "baseline" #@param {type:"string"}
final_val_acc = 0.0 #@param {type:"number"}
final_train_acc = 0.0 #@param {type:"number"}
final_val_loss = 0.0 #@param {type:"number"}
final_train_loss = 0.0 #@param {type:"number"}
total_epochs = 60 #@param {type:"integer"}
notes = "ImprovedCNN with default settings" #@param {type:"string"}

tracker.add_result(
    name=exp_name,
    train_acc=final_train_acc,
    val_acc=final_val_acc,
    train_loss=final_train_loss,
    val_loss=final_val_loss,
    epochs=total_epochs,
    notes=notes
)

print(f"✅ Recorded results for: {exp_name}")
tracker.show_comparison()

In [ ]:
#@title 5.4 Run All Experiments (Batch Mode)
#@markdown Run multiple experiments sequentially

#@markdown ---
#@markdown **Select experiments to run:**
run_baseline = True #@param {type:"boolean"}
run_strong_aug = True #@param {type:"boolean"}
run_higher_lr = True #@param {type:"boolean"}
run_more_wd = False #@param {type:"boolean"}
run_best = True #@param {type:"boolean"}

experiments_to_run = []
if run_baseline:
    experiments_to_run.append(("configs/exp_baseline.yaml", "baseline"))
if run_strong_aug:
    experiments_to_run.append(("configs/exp_strong_aug.yaml", "strong_aug"))
if run_higher_lr:
    experiments_to_run.append(("configs/exp_higher_lr.yaml", "higher_lr"))
if run_more_wd:
    experiments_to_run.append(("configs/exp_more_wd.yaml", "more_wd"))
if run_best:
    experiments_to_run.append(("configs/exp_best.yaml", "best_combined"))

print(f"Will run {len(experiments_to_run)} experiments:")
for config, name in experiments_to_run:
    print(f"  - {name}: {config}")

print("\n" + "="*60)
print("Starting batch experiments...")
print("="*60 + "\n")

for i, (config, name) in enumerate(experiments_to_run, 1):
    print(f"\n{'#'*60}")
    print(f"# EXPERIMENT {i}/{len(experiments_to_run)}: {name}")
    print(f"{'#'*60}\n")
    
    !python train_cls.py {config} --output {name} --print-freq 50 --ckpt-freq 10
    
    print(f"\n✅ Completed: {name}")

print("\n" + "="*60)
print("ALL EXPERIMENTS COMPLETE!")
print("="*60)

---

# 📊 Part 6: Visualize Results

Let's visualize and compare our experiment results.

In [ ]:
#@title 6.1 Launch TensorBoard
#@markdown View training curves interactively

%load_ext tensorboard
%tensorboard --logdir ../logs

In [ ]:
#@title 6.2 Plot Results Comparison
#@markdown Create bar chart comparing experiments

import matplotlib.pyplot as plt
import numpy as np

# Sample data - replace with your actual results
# These are expected values based on improvements
sample_results = {
    "SimpleNet (Baseline)": {"val_acc": 38, "train_acc": 55},
    "+ BatchNorm": {"val_acc": 45, "train_acc": 62},
    "+ Skip Connections": {"val_acc": 50, "train_acc": 68},
    "+ DropPath": {"val_acc": 52, "train_acc": 65},
    "+ Strong Aug": {"val_acc": 54, "train_acc": 60},
    "Best Combined": {"val_acc": 55, "train_acc": 62},
}

# Use actual results if available
if tracker.results:
    results_to_plot = tracker.results
else:
    results_to_plot = sample_results
    print("⚠️ Using sample data. Run experiments to see actual results.\n")

# Create plot
fig, ax = plt.subplots(figsize=(12, 6))

names = list(results_to_plot.keys())
val_accs = [results_to_plot[n].get('val_acc', 0) for n in names]
train_accs = [results_to_plot[n].get('train_acc', 0) for n in names]

x = np.arange(len(names))
width = 0.35

bars1 = ax.bar(x - width/2, val_accs, width, label='Validation Accuracy', color='#2ecc71')
bars2 = ax.bar(x + width/2, train_accs, width, label='Training Accuracy', color='#3498db')

ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('MiniPlaces Experiment Results Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=45, ha='right')
ax.legend()
ax.set_ylim(0, 100)

# Add value labels on bars
for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'{height:.1f}%',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords="offset points",
                ha='center', va='bottom', fontsize=9)

# Add baseline reference line
if val_accs:
    baseline = val_accs[0]
    ax.axhline(y=baseline, color='red', linestyle='--', alpha=0.5, label=f'Baseline ({baseline}%)')

plt.tight_layout()
plt.savefig('experiment_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Plot saved as 'experiment_comparison.png'")

In [ ]:
#@title 6.3 Final Summary
#@markdown Display final summary of all improvements

print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║                    MINIPLACES EXPERIMENT SUMMARY                              ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                               ║
║  KEY IMPROVEMENTS IMPLEMENTED:                                                ║
║  ════════════════════════════                                                 ║
║                                                                               ║
║  1. BATCH NORMALIZATION                                                       ║
║     • Added after each Conv layer, before ReLU                                ║
║     • Enables higher learning rates                                           ║
║     • Acts as regularization                                                  ║
║     → Expected gain: +2-5% accuracy                                           ║
║                                                                               ║
║  2. SKIP CONNECTIONS (ResNet-style)                                           ║
║     • output = F(x) + x                                                       ║
║     • Better gradient flow                                                    ║
║     • Enables deeper networks                                                 ║
║     → Expected gain: +3-5% accuracy                                           ║
║                                                                               ║
║  3. STOCHASTIC DEPTH (DropPath)                                               ║
║     • Randomly drops residual blocks during training                          ║
║     • Linear decay: more dropping in deeper layers                            ║
║     • Regularization for deep networks                                        ║
║     → Expected gain: +1-2% accuracy                                           ║
║                                                                               ║
║  4. DATA AUGMENTATION                                                         ║
║     • Stronger color jitter (0.3 vs 0.15)                                     ║
║     • More rotation (25° vs 15°)                                              ║
║     • Larger initial size for crop variety                                    ║
║     → Expected gain: +1-2% accuracy                                           ║
║                                                                               ║
║  5. TRAINING OPTIMIZATIONS                                                    ║
║     • Extended warmup (3-5 epochs)                                            ║
║     • Optimized weight decay (3e-4)                                           ║
║     • Cosine learning rate schedule                                           ║
║     → Expected gain: +1-2% accuracy                                           ║
║                                                                               ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  TOTAL EXPECTED IMPROVEMENT: +10-15% over baseline SimpleNet                  ║
╚══════════════════════════════════════════════════════════════════════════════╝
""")

# Show actual results if available
tracker.show_comparison()

In [ ]:
#@title 6.4 Save Results to Google Drive
#@markdown Save all results and checkpoints to Drive

import shutil

# Copy logs and results
src_logs = "../logs"
dst_logs = f"{WORK_DIR}/logs"

if os.path.exists(src_logs):
    if os.path.exists(dst_logs):
        shutil.rmtree(dst_logs)
    shutil.copytree(src_logs, dst_logs)
    print(f"✅ Logs saved to: {dst_logs}")

# Copy results JSON
if os.path.exists("experiment_results.json"):
    shutil.copy("experiment_results.json", WORK_DIR)
    print(f"✅ Results saved to: {WORK_DIR}/experiment_results.json")

# Copy plot
if os.path.exists("experiment_comparison.png"):
    shutil.copy("experiment_comparison.png", WORK_DIR)
    print(f"✅ Plot saved to: {WORK_DIR}/experiment_comparison.png")

print(f"\n📁 All results saved to: {WORK_DIR}")

---

# 📝 Quick Reference

## Commands to Run Experiments

```bash
# Baseline (ImprovedCNN with default settings)
python train_cls.py configs/exp_baseline.yaml --output baseline

# With stronger augmentation
python train_cls.py configs/exp_strong_aug.yaml --output strong_aug

# With higher learning rate
python train_cls.py configs/exp_higher_lr.yaml --output higher_lr

# Best combined settings
python train_cls.py configs/exp_best.yaml --output best_combined
```

## Switching Models in modeling.py

```python
# At the end of modeling.py, change:
default_cnn_model = SimpleNet      # Original baseline
default_cnn_model = SimpleNetBN    # + BatchNorm
default_cnn_model = ImprovedCNN    # Full ResNet-style (RECOMMENDED)
```

## Key Lecture Concepts

| Technique | Why It Helps |
|-----------|-------------|
| Batch Normalization | Stabilizes training, implicit regularization |
| Skip Connections | Better gradient flow, enables deeper networks |
| Stochastic Depth | Regularization, implicit ensemble |
| He Initialization | Proper weight scaling for ReLU |
| Cosine LR Schedule | Smooth decay, better convergence |
| Data Augmentation | Expands effective training set |